# Planning a flux growth in an unexplored ternary system with $G_{\mathrm{Liquid}}$

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willwerj/gliquid_python/blob/main/notebooks/colab_demo.ipynb)

$G_{\mathrm{Liquid}}$ is a DFT-referenced liquid free energy which when compared to the DFT convex hull (MP GGA), recreates the liquidus curve for the sytem. Where an assessed phase diagram exists (from MPDS), it fits the parameters to the measured liquidus. Where none exists, a machine-learning model predicts them. We can use these together to interpolate any ternary liquidus surface from binaries in our 60-element dataset.

This notebook follows one question through that pipeline: **could Ru₂Si₃ single crystals be grown from a gallium flux?** The answer lies on the Ga–Ru–Si liquidus, and one of its three binaries, Ga–Ru, has no previously-assessed phase diagram

## Properties of Ru₂Si₃ 

Ru₂Si₃ has interesting properties which may warrant growing this as a single crystal at lower temperatures, possibly achievable through a low-melting flux such as Ga

- **Semiconductor.** Ru₂Si₃ has a direct band gap of 0.84 eV, measured on films and on a single crystal. The matching wavelength, near 1.5 µm, is where glass fibres absorb least ([Lenssen et al., 2001](https://doi.org/10.1063/1.1402965)). Optical spectra and band structure: [Henrion et al., 2000](https://doi.org/10.1016/S0040-6090(99)00958-X).
- **High-temperature thermoelectric.** Extrapolating from undoped crystals, doped Ru₂Si₃ was predicted to outperform silicon–germanium ([Vining, 1992](https://doi.org/10.1063/1.41793)). Mn-doped crystals reach a Seebeck coefficient of 400 µV/K near 500 K ([Krivosheev et al., 2006](https://doi.org/10.1134/S1063782606010039)), and the figure of merit along [010] is more than four times that along [100] at 900 K ([Simkin, Hayashi and Inui, 2005](https://doi.org/10.1016/j.intermet.2005.04.006)).

Ru₂Si₃ melts congruently near 1690 °C, and single crystals have been grown by optical floating zone ([Souptel et al., 2002](https://doi.org/10.1016/S0022-0248(02)01686-X); [Krivosheev et al., 2006](https://doi.org/10.1134/S1063782606010039)). According to the asssesed Ru-Si phase diagram ([Long et al., 2009](https://doi.org/10.1016/j.jallcom.2009.01.079); [MPDS: C120159](https://mpds.io/entry/C120159)), the lowest temperature where single crystalline Ru₂Si₃ can be grown from a self-flux is ~1300 °C

<img src="../assets/ru_si_assessed_pd.png" style="max-width: 700px; width: 100%; height: auto;">

## Growing Ru₂Si₃ from a flux

Suppose we want to grow Ru₂Si₃ below 1200 °C. A low-melting metal flux makes this possible: a crystal dissolved in the flux grows far below its own melting point ([Kanatzidis, Pöttgen and Jeitschko, 2005](https://doi.org/10.1002/anie.200462170); [Phelan et al., 2012](https://doi.org/10.1021/cm2019873)). Gallium, which melts at 30 °C, is a natural candidate.

A Ga flux works only if Ru₂Si₃, and not a Ga–Ru compound, is the primary phase (the first solid to crystallize) from a Ga-rich liquid at the growth temperature. The Ga–Ru–Si liquidus answers that question, but it has never been assessed, and only two of its three binaries (Ru–Si and Ga–Si) have assessed diagrams. Without that surface, finding a working flux composition means a costly series of trial growths.

$G_{\mathrm{Liquid}}$ builds the missing surface from what is known:

**1.** Fit Ru–Si and Ga–Si to their MPDS liquidus,

**2.** Predict Ga–Ru with the ML model,

**3.** Interpolate the Ga–Ru–Si liquidus from the binaries,

**4.** Revise the Ga–Ru edge against measurements, and interpolate again


## 1a) Clone the data and install the package

The per-system data (digitized MPDS diagrams and DFT convex hulls) lives in the repository's `cache/` directory, so a shallow clone fetches it. Colab already provides the notebook tooling, xgboost (for the ML model) and ipywidgets (for the convex hull editor), so the package itself needs no extras.

In [ ]:
!git clone --depth 1 https://github.com/willwerj/gliquid_python.git
!pip install -q "gliquid==0.3.2" "pymatgen<2026"
from google.colab import output
output.enable_custom_widget_manager()

## 1b) Point the cache at cloned $G_{\mathrm{Liquid}}$ data

In [ ]:
import gliquid.config as cfg

# Point GLiquid at the cloned repository's cache store
cfg.set_cache_dir("/content/gliquid_python/cache")
print(f"Cache directory: {cfg.cache_dir}")

## 1c) Fit the two assessed binary systems

RuSi₂ is on the DFT hull but not on the assessed Ru–Si diagram, so we remove it from the DFT convex hull prior to fitting. The plot compares the fitted liquidus (solid line) with the assessed one (dashed); the solid phases come from the edited hull.

In [ ]:
from gliquid.binary import BinaryLiquid, BLPlotter
from gliquid.hull_editor import ConvexHullEditor

ru_si = BinaryLiquid.from_cache("Ru-Si", param_format="comb-exp")

# Remove the Si2Ru entry from the DFT convex hull, since it does not appear as stable phase in the assessed phase diagram
# This is not done in the typical fitting workflow, but is done here to properly attribute the ternary liquidus surface
editor = ConvexHullEditor(ru_si)
for entry in reversed(editor.entries):  # reversed: removing an entry renumbers the ones after it
    if entry["name"] == "Si2Ru":
        editor.remove_entry(entry["index"])
editor.apply()
print("Ru-Si solid phases:", ", ".join(p.name for p in ru_si.phases if p.name != "L"))


def fit(bl):
    """Fit ``bl`` to its MPDS liquidus and report the best of two optimizations."""
    results = bl.fit_parameters(n_opts=2)
    best = min(results, key=lambda r: r.get("mae", float("inf")), default={})
    print(f"{'-'.join(bl.components)}: liquidus MAE {best['mae']:.0f} K ({best['mape']:.1f} %)")
    print("  parameters [L0_a, L0_b, L1_a, L1_b]:", [round(p, 2) for p in bl.get_params()])


fit(ru_si)
BLPlotter(ru_si).show("fit+liq")

Ga–Si is a simple eutectic-forming system with no compounds, Here, we fit from the assessment published in the 1990 ASM Handbook ([Olesinski, Kanani, and Abbaschian, 1990](https://doi.org/10.1002/adma.19910031215); [MPDS: C901130](https://mpds.io/#entry/C901130)).

In [ ]:
ga_si = BinaryLiquid.from_cache("Ga-Si", param_format="comb-exp")
fit(ga_si)
BLPlotter(ga_si).show("fit+liq")

## 2) Predict the unexplored binary system: Ga–Ru

As of 10/13/2026, there has been no published Ga–Ru assessment in the literature. Here, we use $G_{\mathrm{Liquid}}$'s ML model to predict the DFT-referenced liquid free energy for the system. By comparing this predicted liquid free energy to the DFT *T* = 0 K convex hull, we can predict the full phase diagram.

In [ ]:
from gliquid.production_model_runner import ProductionModelRunner

runner = ProductionModelRunner()
pred_params = runner.predict_system("Ga-Ru")
print("Predicted parameters [L0_a, L0_b, L1_a]:", [round(p, 2) for p in pred_params])

ga_ru = BinaryLiquid.from_cache("Ga-Ru", param_format="comb-exp")
ga_ru.update_params(list(pred_params) + [0])  # L1_b must be input even when not used for gliquid v0.3.0.rc1
BLPlotter(ga_ru).show("pred")

## 3) Interpolate the Ga–Ru–Si liquidus surface

The two fitted edges (Ru-Si, Ga-Si) and the predicted Ga–Ru can be used to build ternary liquid model, under the assumption that the mixing character is dominated by the pairwise interactions, which are interpolated in accordance with the Muggianu mixing scheme.


We can highlight the Ru₂Si₃ field — the part of the liquidus where Ru₂Si₃ is the primary phase — to see which compositions and temperatures the phase is predicted to grow at near-equilibrium conditions

*Clicking another phase in the left legend highlights its field instead, and clicking it again restores the full surface.*



In [ ]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from gliquid.liquidus_sheets import LiquidusSheets
from gliquid.plotting.export import side_by_side_html
from gliquid.plotting.ternary_labels import orient_labels
from gliquid.ternary import TLIPlotter, TernaryLiquidInterpolation

RU2SI3 = "Si3Ru2"  # pymatgen's formula for Ru2Si3

def show_two_views(ternary, phase=RU2SI3):
    """Draw the liquidus with ``phase``'s field highlighted, rotatable and seen from above."""
    plotter = TLIPlotter(ternary)
    rotatable = plotter.get_plot("tx", highlight_phase=phase)

    above = plotter.get_plot("tx", highlight_phase=phase, interactive_legend=False)
    above.layout.annotations = ()  # the temperature axis title
    aspect, zoom = above.layout.scene.aspectratio, 1.5
    above.update_layout(
        showlegend=False,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            dragmode=False,
            hovermode=False,
            xaxis_visible=False,
            yaxis_visible=False,
            zaxis_visible=False,
            aspectratio=dict(x=zoom * aspect.x, y=zoom * aspect.y, z=aspect.z),
            camera=dict(eye=dict(x=0, y=0, z=1), up=dict(x=0, y=1, z=0)),
        ),
    )
    orient_labels(above)  # composition labels upright for the top-down camera

    display(HTML(side_by_side_html(
        [rotatable, above],
        configs=[None, {"displayModeBar": False, "scrollZoom": False}],
    )))


def ga_rich_liquidus(ternary, ga_fractions=(0.99, 0.98, 0.95, 0.90)):
    """Print the liquidus of gallium holding dissolved Ru2Si3, and the solid that forms first."""
    sheets = LiquidusSheets.from_interpolation(ternary)  # each solid's liquidus temperature
    ga = np.array(ga_fractions)
    fractions = {"Ga": ga, "Ru": 0.4 * (1 - ga), "Si": 0.6 * (1 - ga)}
    temps = sheets(np.column_stack([fractions[c] for c in ternary.components[1:]])) - 273.15
    print("Gallium with dissolved Ru2Si3 (Ru:Si = 2:3):")
    for x_ga, row in zip(ga, temps):
        print(f"  {100 * (1 - x_ga):3.0f} at.% Ru2Si3: liquidus {row.max():5.0f} °C, "
              f"{sheets.phases[row.argmax()]} crystallizes first")


predicted_ternary = TernaryLiquidInterpolation.from_binaries([ru_si, ga_si, ga_ru])
show_two_views(predicted_ternary)
ga_rich_liquidus(predicted_ternary)

On the predicted surface the Ru₂Si₃ field runs from the Ru–Si edge into the Ga corner, so gallium holding anywhere from 0.5 to 15 at.% Ru₂Si₃ crystallizes Ru₂Si₃ first. Up to about 5 at.%, its liquidus lies below 930 °C, where the transformation begins, so a growth there would form the orthorhombic phase directly.

All of this rests on the predicted Ga–Ru edge, which sets how much ruthenium a gallium-rich liquid holds.

## 4a) Revise the Ga–Ru system prediction against measurements

Following the initial Ga-Ru prediction, our collaborators at Baylor University performed differential scanning calorimetry (DSC) measurements on a few different compositions  

To match the model to observed reactions, we assign Ga₃Ru and GaRu an entropy (−1.5 and +7.6 J/mol-atom/K, with the Gibbs hull evaluated at 1108 °C), then refit the liquid parameters to the measured liquidus points. The cell below applies that notebook's result and marks the measured liquidus. (Full workflow: [GaRu_adjustment.ipynb](https://colab.research.google.com/github/willwerj/gliquid_python/blob/main/notebooks/GaRu_adjustment.ipynb)). 

In [ ]:
editor = ConvexHullEditor(ga_ru)
editor.set_temperature(1108 + 273.15)  # evaluate the Gibbs hull at the measured liquidus temperature (K)
editor.modify_entry(editor.find_entry(name="Ga3Ru"), new_entropy=-1.5, units="J/mol")  # J/mol-atom/K
editor.modify_entry(editor.find_entry(name="GaRu"), new_entropy=7.6, units="J/mol")  # J/mol-atom/K
editor.apply()
ga_ru.update_params([-101428.21, -12.679, 23601.63, 0.0])  # parameters fitted to DSC measurements in GaRu_adjustment.ipynb

measured = [[5, 1008.8], [15, 1108.1]]  # DSC liquidus [at.% Ru, °C], averaged per composition
figure = BLPlotter(ga_ru).get_plot("pred")
figure.add_trace(go.Scatter(x=[x for x, _ in measured], y=[t for _, t in measured], mode="markers",
                            name="Liquidus measured (DSC)",
                            marker=dict(color="cornflowerblue", size=12, symbol="square",
                                        line=dict(width=1, color="black"))))
figure.show()

## 4b) Revised Ga–Ru–Si liquidus surface
The same construction, with the revised Ga–Ru prediction. `from_binaries` carries its edited phases into the ternary as user-defined phases with their entropies

In [ ]:
revised_ternary = TernaryLiquidInterpolation.from_binaries([ru_si, ga_si, ga_ru])

solids = revised_ternary.get_ternary_form_en(revised_ternary.components)  # x0 = x(Ru), x1 = x(Si)
user = solids[solids["Phase Name"].isin(revised_ternary.user_phase_names())]
print("User-defined solid phases in the ternary hull (S in J/mol-atom/K, H in J/mol-atom):")
print(user.rename(columns={"x0": "x(Ru)", "x1": "x(Si)"}).sort_values(["x(Si)", "x(Ru)"]).round(3).to_string(index=False))

show_two_views(revised_ternary)
ga_rich_liquidus(revised_ternary)

The measured Ga–Ru edge raises the Ga-rich liquidus by 120 to 210 K, most in the most dilute solutions. Ru₂Si₃ still crystallizes first along the whole line, but the liquidus now stays below 930 °C only up to about 1.5 at.% Ru₂Si₃. From the hover tool, we can see that we need a minimum flux composition of 91.5 at.% Ga to grow Ru₂Si₃ below 1200 °C, with the growth temperature sharply declining as additional Ga is added.

# In-depth workflow notebooks

To see any of the steps here in greater detail, check out these notebooks.

If running on Google Colab, uncomment the first code block in each notebook to set up the Colab environment.

### Fiting $G_{\mathrm{Liquid}}$ to an assessed binary system with MPDS data
[![fitting_demo.ipynb](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willwerj/gliquid_python/blob/main/notebooks/fitting_demo.ipynb) fitting_demo.ipynb

### Interpolating a ternary liqudius surface from fitted binaries
[![interpolation_demo.ipynb](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willwerj/gliquid_python/blob/main/notebooks/interpolation_demo.ipynb) interpolation_demo.ipynb

### Adjusting liquidus predictions from characterized phases, solidus and liquidus points (Ga-Ru and Pb-Tb systems)
[![GaRu_adjustment.ipynb](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willwerj/gliquid_python/blob/main/notebooks/GaRu_adjustment.ipynb) GaRu_adjustment.ipynb

[![PbTb_adjustment.ipynb](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willwerj/gliquid_python/blob/main/notebooks/PbTb_adjustment.ipynb) PbTb_adjustment.ipynb
